# RNN and LSTM — Sequence Models

MLPs and CNNs process fixed-size inputs with no memory of previous inputs.
RNNs process **sequences** — each step takes the current input + a hidden state from the previous step.

**Where you still see RNNs/LSTMs:** time series forecasting, speech recognition, music generation.
Transformers have replaced them for NLP, but understanding LSTMs helps understand attention.

In [ ]:
import torch
import torch.nn as nn

# --- RNN basics ---
# input_size  = features per timestep
# hidden_size = size of hidden state (memory)
# num_layers  = stacked RNN layers

rnn = nn.RNN(input_size=10, hidden_size=32, num_layers=1, batch_first=True)

# batch_first=True → input shape [batch, seq_len, features]
#                    (default is [seq_len, batch, features] — confusing, always use batch_first)

batch, seq_len, features = 4, 20, 10
x = torch.randn(batch, seq_len, features)

output, h_n = rnn(x)
# output: [batch, seq_len, hidden_size] — hidden state at EVERY timestep
# h_n:    [num_layers, batch, hidden_size] — hidden state at LAST timestep only

print('Input shape: ', x.shape)
print('Output shape:', output.shape)   # [4, 20, 32]
print('h_n shape:   ', h_n.shape)      # [1, 4, 32]

## LSTM — Long Short-Term Memory

RNNs have a **vanishing gradient problem** — gradients shrink to zero over long sequences, so the model forgets early inputs.

LSTM solves this with a **cell state** (long-term memory) + **gates** that control what to remember/forget:
- **Forget gate:** what to discard from cell state
- **Input gate:** what new info to store
- **Output gate:** what to output as hidden state

This lets LSTMs remember information hundreds of timesteps back.

In [ ]:
lstm = nn.LSTM(input_size=10, hidden_size=32, num_layers=2, batch_first=True, dropout=0.1)

x = torch.randn(4, 20, 10)   # [batch=4, seq=20, features=10]

output, (h_n, c_n) = lstm(x)
# output: [batch, seq_len, hidden_size]
# h_n:    [num_layers, batch, hidden_size] — hidden state (short-term memory)
# c_n:    [num_layers, batch, hidden_size] — cell state (long-term memory)

print('Output shape:', output.shape)   # [4, 20, 32]
print('h_n shape:   ', h_n.shape)      # [2, 4, 32]  — 2 because num_layers=2
print('c_n shape:   ', c_n.shape)      # [2, 4, 32]

In [ ]:
# --- Sentiment Classification with LSTM ---
# Task: classify sequences of numbers as positive or negative trend

torch.manual_seed(42)

class SentimentLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(hidden_size, 1)   # binary output

    def forward(self, x):
        output, (h_n, _) = self.lstm(x)
        # Use last timestep's hidden state for classification
        last_hidden = h_n[-1]             # [batch, hidden_size] — last layer
        return self.fc(last_hidden)        # [batch, 1]


# Generate synthetic data: increasing trend = label 1, decreasing = label 0
def make_data(n=500, seq_len=20):
    X, y = [], []
    for _ in range(n):
        if torch.rand(1) > 0.5:
            seq = torch.cumsum(torch.rand(seq_len, 1) * 0.1, dim=0)    # upward
            label = 1.0
        else:
            seq = -torch.cumsum(torch.rand(seq_len, 1) * 0.1, dim=0)   # downward
            label = 0.0
        X.append(seq)
        y.append(label)
    return torch.stack(X), torch.tensor(y)


X, y = make_data()
print('X shape:', X.shape)   # [500, 20, 1]
print('y shape:', y.shape)   # [500]

In [ ]:
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = SentimentLSTM().to(device)
loss_fn   = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

X, y = X.to(device), y.to(device)

batch_size = 32
for epoch in range(10):
    model.train()
    total_loss, correct = 0, 0
    for i in range(0, len(X), batch_size):
        X_b, y_b = X[i:i+batch_size], y[i:i+batch_size]
        logits = model(X_b).squeeze()
        loss   = loss_fn(logits, y_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += ((logits > 0).float() == y_b).sum().item()

    if epoch % 2 == 0:
        acc = correct / len(X)
        print(f'Epoch {epoch+1} | loss: {total_loss/len(X)*batch_size:.4f} | acc: {acc:.3f}')